# 01 — Embed & Search
1. Filter your protein FASTAs (200–500 AA)
2. Run HMMER prefilter (PF01553 domain)
3. Embed HMMER hits with ESM2-650M
4. Build FAISS KNN index
5. Query with reference probes

In [ ]:
import os, sys
from pathlib import Path

# ── CONFIG (edit here) ────────────────────────────────────────────────────────
PROJECT_ROOT  = '/content/drive/MyDrive/acyltransferase'  # or local path
YOUR_FASTA    = 'data/proteins_raw/all_proteins.faa'       # your input FASTA
DEVICE        = 'cuda'   # 'cuda' on Colab GPU runtime, 'cpu' otherwise
BATCH_SIZE    = 64       # reduce to 32 if OOM on GPU
KNN_K         = 20       # neighbours per query probe
# ─────────────────────────────────────────────────────────────────────────────

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

from acyltransferase.config import load_config
cfg = load_config('config.yaml')

# override device from cell config
cfg.embedding['device'] = DEVICE
cfg.embedding['batch_size'] = BATCH_SIZE

In [ ]:
# 1. Filter proteins by length
from acyltransferase.utils import read_fasta, write_fasta, filter_by_length, deduplicate_fasta

raw_records = read_fasta(YOUR_FASTA)
filtered    = deduplicate_fasta(filter_by_length(
    raw_records,
    min_len=cfg.filter['min_length'],
    max_len=cfg.filter['max_length'],
))

filtered_faa = 'data/proteins_filtered.faa'
write_fasta(filtered, filtered_faa)
print(f'Raw: {len(raw_records)}  →  After filter+dedup: {len(filtered)}')

In [ ]:
# 2. HMMER prefilter
from acyltransferase.search import run_hmmer

hmm_path  = f"data/hmm_profiles/{cfg.hmmer['profiles'][0]}.hmm"
hits_faa  = run_hmmer(
    fasta_path  = filtered_faa,
    hmm_path    = hmm_path,
    output_dir  = 'data/hmmer_hits',
    evalue      = cfg.hmmer['evalue'],
    cpu         = cfg.hmmer['cpu'],
)

hits = read_fasta(hits_faa)
print(f'HMMER hits: {len(hits)} sequences → {hits_faa}')

In [ ]:
# 3. Embed hits with ESM2
from acyltransferase.embed import embed_sequences, save_embeddings

os.makedirs('data/embeddings', exist_ok=True)
emb_npy  = 'data/embeddings/hits.npy'
emb_ids  = 'data/embeddings/hits_ids.txt'

if not Path(emb_npy).exists():
    print(f'Embedding {len(hits)} sequences on {DEVICE} ...')
    embeddings, ids = embed_sequences(
        hits,
        model_name = cfg.embedding['model'],
        batch_size = cfg.embedding['batch_size'],
        device     = cfg.embedding['device'],
        layer      = cfg.embedding['layer'],
    )
    save_embeddings(embeddings, ids, emb_npy, emb_ids)
    print(f'Saved: {embeddings.shape}')
else:
    print(f'Embeddings already exist: {emb_npy}')

In [ ]:
# Embed reference probes
probes_faa = 'data/query_sequences/reference_probes.faa'
ref_npy    = 'data/embeddings/ref.npy'
ref_ids    = 'data/embeddings/ref_ids.txt'

if not Path(ref_npy).exists():
    probe_records = read_fasta(probes_faa)
    ref_emb, rids = embed_sequences(
        probe_records,
        model_name = cfg.embedding['model'],
        batch_size = 4,  # small batch for reference probes
        device     = cfg.embedding['device'],
        layer      = cfg.embedding['layer'],
    )
    save_embeddings(ref_emb, rids, ref_npy, ref_ids)
    print(f'Reference embeddings: {ref_emb.shape}')

In [ ]:
# 4. Build FAISS KNN index
import numpy as np
from acyltransferase.search import build_knn_index

embeddings = np.load(emb_npy)
ids        = Path(emb_ids).read_text().splitlines()

os.makedirs('data/knn_index', exist_ok=True)
build_knn_index(embeddings, ids,
                index_path  = 'data/knn_index/hits.index',
                id_map_path = 'data/knn_index/id_map.tsv')
print(f'FAISS index built: {len(ids)} vectors')

In [ ]:
# 5. Query: find nearest neighbours for each reference probe
from acyltransferase.search import query_knn

probe_records = read_fasta(probes_faa)
knn_hits = query_knn(
    query_sequences = probe_records,
    index_path      = 'data/knn_index/hits.index',
    id_map_path     = 'data/knn_index/id_map.tsv',
    k               = KNN_K,
    embed_kwargs    = {
        'model_name': cfg.embedding['model'],
        'batch_size': 4,
        'device':     cfg.embedding['device'],
    }
)

knn_hits.to_csv('results/knn_hits.tsv', sep='\t', index=False)
print(knn_hits.head(10))

In [ ]:
# Quick distance distribution plot
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4))
for qid, grp in knn_hits.groupby('query_id'):
    ax.plot(grp['rank'], grp['l2_dist'], alpha=0.6, linewidth=1,
            label=qid.split('|')[0])
ax.set_xlabel('KNN rank'); ax.set_ylabel('L2 distance')
ax.set_title('KNN distance per reference probe'); ax.legend(fontsize=6, ncol=4)
plt.tight_layout(); plt.show()

## ✓ Done
Next: **02_cluster.ipynb** — Leiden clustering and t-SNE visualisation.